# 10 — SHAP + UTPM Lineer İndeks + Moran's I + Jenks (Hafta 12-13)

**Tezsel sonuç ürünü.** Bu notebook 4 bağlantılı analiz yapar:

1. **SHAP TreeExplainer** — RF tahminlerinin yorumlanabilir attribusyonu (her feature'ın her hücreye katkısı)
2. **UTPM lineer indeksi** — SHAP global importance ağırlıklarıyla z-skor toplam → 0-100 ölçek
3. **Moran's I + LISA** — UTPM mekânsal kümeli mi? Local Moran ile her hücreye HH/LL/HL/LH cluster ataması
4. **Jenks Natural Breaks** — UTPM'i 5 sınıfa böl (Çok serin / Serin / Orta / Sıcak / Çok sıcak) → karar destek katmanı

**Önkoşul:** `grid_30m_modeling.gpkg`, `results/rf_model.pkl`, `grid_30m_persistence.gpkg`.

**Çıktılar:**
- `data/processed/grid_30m_utpm.gpkg` — ana sonuç tablosu
- `results/shap_values.npy`, `results/utpm_analysis.json`
- `figures/10_*.png`
- `tables/10_utpm_*.csv`

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap

from src.config import (
    DATA_GRID, DATA_PROCESSED, RESULTS, FIGURES, TABLES,
    GRID_30M_MODELING, MODEL_PKL, PERSISTENCE_GPKG,
    SELECTED_FEATURES, TARGET_COLUMN, CRS_PROJECTED,
    SHAP_VALUES_NPY, UTPM_GPKG, UTPM_RESULTS_JSON,
    SHAP_SAMPLE_N, MORAN_K_NEIGHBORS, JENKS_K_CLASSES,
    UTPM_SIGN_FLIPS,
)
from src.modeling import prepare_modeling_matrix
from src.utpm import (
    shap_tree_explainer, shap_global_importance,
    compute_utpm_index, normalize_utpm,
    compute_moran_i, compute_local_moran, jenks_classify,
)

%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

In [ ]:
# --- Yükle ---
grid = gpd.read_file(GRID_30M_MODELING)
X, y, feat_names = prepare_modeling_matrix(
    grid, features=SELECTED_FEATURES, target=TARGET_COLUMN,
    fillna_zero=True, add_saturated_flag=True,
)
model = joblib.load(MODEL_PKL)
print(f"Grid: {len(grid):,}  X: {X.shape}  RF: {model.n_estimators} trees")

In [ ]:
# --- SHAP TreeExplainer (5K sample) ---
print(f"SHAP hesaplanıyor (sample {SHAP_SAMPLE_N}, ~5-15 dk)...")
sample_X, shap_values, explainer = shap_tree_explainer(
    model, X, sample_n=SHAP_SAMPLE_N,
)
print(f"  shape: {shap_values.shape}")

shap_imp = shap_global_importance(shap_values, feat_names)
print("\nGlobal SHAP importance:")
print(shap_imp.to_string(index=False))

In [ ]:
# --- SHAP Beeswarm + Feature Dependence ---
fig = plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, sample_X, plot_type="dot",
                   feature_names=feat_names, show=False)
plt.tight_layout()
plt.savefig(FIGURES / "10_shap_beeswarm.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
# --- UTPM ağırlıkları (SHAP-based) ---
utpm_features = [f for f in feat_names if f != "is_dtc_saturated"]
z_features = [f"{f}_z" for f in utpm_features]

shap_dict = dict(zip(shap_imp["feature"], shap_imp["shap_normalized"]))
weights = np.array([shap_dict[f] for f in utpm_features])
weights = weights / weights.sum()

weights_df = pd.DataFrame({
    "feature": utpm_features,
    "weight": weights.round(4),
    "sign_flip": [UTPM_SIGN_FLIPS.get(f"{f}_z", 1.0) for f in utpm_features],
})
print("UTPM ağırlıkları (SHAP-based, is_dtc_saturated hariç):")
print(weights_df.sort_values("weight", ascending=False).to_string(index=False))

In [ ]:
# --- UTPM hesabı ---
utpm_raw = compute_utpm_index(
    grid, z_features=z_features, weights=weights,
    sign_flips=UTPM_SIGN_FLIPS,
)
grid["utpm_raw"] = utpm_raw
grid["utpm_score"] = normalize_utpm(utpm_raw, method="minmax") * 100

print(f"UTPM skoru özet (0-100):")
print(grid['utpm_score'].describe().round(2))

r_utpm_lst = float(grid[['utpm_score', 'lst_mean']].corr().iloc[0, 1])
print(f"\nUTPM × LST korelasyonu: r = {r_utpm_lst:.3f}")
print("(Yüksek r = lineer indeks RF'in non-lineer sinyalini iyi yansıtmış)")

In [ ]:
# --- Moran's I (Global) ---
print(f"Global Moran's I (k={MORAN_K_NEIGHBORS} en yakın komşu)...")
moran = compute_moran_i(grid['utpm_score'].to_numpy(), grid.geometry,
                         k_neighbors=MORAN_K_NEIGHBORS)
print(f"  I = {moran['I']:.4f}  (E[I] = {moran['expected_I']:.4f})")
print(f"  z-score = {moran['z_score']:.2f}")
print(f"  p-value = {moran['p_value']:.4f}")
print(f"\nYorum: I > 0 + p < 0.05 → UTPM mekânsal kümeli (UHI hipotezi destekleyici)")

In [ ]:
# --- LISA (Local Moran) ---
print("LISA hesaplanıyor (~1-2 dk)...")
local_m = compute_local_moran(grid['utpm_score'].to_numpy(), grid.geometry,
                                k_neighbors=MORAN_K_NEIGHBORS)
grid['local_I'] = local_m['Is'].values
grid['lisa_q'] = local_m['q'].values
grid['lisa_p'] = local_m['p_sim'].values
grid['lisa_cluster'] = local_m['cluster_label'].values

print("\nLISA cluster dağılımı:")
print(grid['lisa_cluster'].value_counts().to_string())
print("\nHH = High-High (sıcak hücre, sıcak komşular — UHI çekirdeği)")
print("LL = Low-Low (serin küme — soğuk ada)")
print("HL/LH = uyumsuz (yerel anomali)")
print("NS = Not significant")

In [ ]:
# --- Jenks 5 sınıf ---
classes, bins = jenks_classify(grid['utpm_score'].to_numpy(), k=JENKS_K_CLASSES)
grid['utpm_class'] = classes

class_labels = ['Çok serin', 'Serin', 'Orta', 'Sıcak', 'Çok sıcak']
print(f"Jenks Natural Breaks ({JENKS_K_CLASSES} sınıf):")
for i, (a, b) in enumerate(zip(bins[:-1], bins[1:])):
    n = (classes == i).sum()
    label = class_labels[i] if i < len(class_labels) else f"Sınıf {i}"
    pct = 100 * n / len(grid)
    print(f"  {label:11s} ({a:.1f} – {b:.1f}):  {n:>6,} hücre  (%{pct:.1f})")

In [ ]:
# --- Persistence × UTPM bağlantısı ---
persist = gpd.read_file(PERSISTENCE_GPKG)
merged = grid.merge(
    persist[['cell_id', 'years_in_top_quartile']],
    on='cell_id', how='left',
)

ph_mask = merged['years_in_top_quartile'] == 5
print(f"Persistent HOT hücreler ({ph_mask.sum():,}):")
print(f"  UTPM medyan: {merged.loc[ph_mask, 'utpm_score'].median():.2f}")
print(f"  UTPM mean:   {merged.loc[ph_mask, 'utpm_score'].mean():.2f}")

r_persist = float(merged[['utpm_score', 'years_in_top_quartile']].corr().iloc[0, 1])
print(f"\nUTPM × persistence: r = {r_persist:.3f}")
print("(Yüksek r = lineer indeks 5-yıl persistent hot pattern'i ortaya çıkarıyor)")

In [ ]:
# --- Görselleştirme: 4 panel final harita ---
fig, axes = plt.subplots(2, 2, figsize=(15, 13))
boundary = gpd.read_file(DATA_GRID / 'pilot_boundary.gpkg')

# 1. UTPM continuous
ax = axes[0, 0]
grid.plot(column='utpm_score', cmap='inferno', ax=ax, linewidth=0,
          legend=True, legend_kwds={'label': 'UTPM skor (0-100)', 'shrink': 0.6})
boundary.to_crs(CRS_PROJECTED).boundary.plot(ax=ax, color='#0a9396', linewidth=0.8)
ax.set_title(f'UTPM continuous (n={len(grid):,})')
ax.set_aspect('equal')
ax.ticklabel_format(style='plain')

# 2. Jenks 5 sınıf
ax = axes[0, 1]
from matplotlib.colors import ListedColormap
jenks_colors = ['#264653', '#2A9D8F', '#E9C46A', '#F4A261', '#E76F51']
jenks_cmap = ListedColormap(jenks_colors)
grid.plot(column='utpm_class', categorical=True, cmap=jenks_cmap, ax=ax,
          linewidth=0, legend=True,
          legend_kwds={'labels': class_labels, 'loc': 'upper right'})
boundary.to_crs(CRS_PROJECTED).boundary.plot(ax=ax, color='#0a9396', linewidth=0.8)
ax.set_title('Jenks 5-sınıf (karar destek)')
ax.set_aspect('equal')
ax.ticklabel_format(style='plain')

# 3. LISA cluster
ax = axes[1, 0]
lisa_colors = {'HH': '#d7191c', 'LL': '#2c7bb6', 'HL': '#fdae61',
               'LH': '#abd9e9', 'NS': '#dddddd'}
for cluster, color in lisa_colors.items():
    sub = grid[grid['lisa_cluster'] == cluster]
    if len(sub) > 0:
        sub.plot(ax=ax, color=color, linewidth=0,
                  label=f'{cluster} ({len(sub):,})')
boundary.to_crs(CRS_PROJECTED).boundary.plot(ax=ax, color='#0a9396', linewidth=0.8)
ax.legend(loc='upper right', fontsize=8)
ax.set_title('LISA Local Moran cluster')
ax.set_aspect('equal')
ax.ticklabel_format(style='plain')

# 4. UTPM ağırlıkları bar
ax = axes[1, 1]
wd = weights_df.sort_values('weight')
y_pos = np.arange(len(wd))
ax.barh(y_pos, wd['weight'], color='#2A9D8F', edgecolor='black')
ax.set_yticks(y_pos)
ax.set_yticklabels(wd['feature'], fontsize=9)
ax.set_xlabel('UTPM ağırlığı (SHAP-normalize)')
ax.set_title(f'UTPM ağırlıkları + Moran I={moran["I"]:.3f} (p={moran["p_value"]:.3f})')
for i, v in enumerate(wd['weight']):
    ax.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES / '10_utpm_overview.png', dpi=140, bbox_inches='tight')
plt.show()

In [ ]:
# --- Kaydet ---
out_cols = ['cell_id', 'geometry', TARGET_COLUMN,
            'utpm_raw', 'utpm_score', 'utpm_class',
            'local_I', 'lisa_q', 'lisa_p', 'lisa_cluster']
grid[out_cols].to_file(UTPM_GPKG, driver='GPKG')
print(f'Kaydedildi: {UTPM_GPKG.name} ({UTPM_GPKG.stat().st_size/1024/1024:.2f} MB)')

np.save(SHAP_VALUES_NPY, shap_values)
print(f'Kaydedildi: {SHAP_VALUES_NPY.name}')

results = {
    'shap_importance': shap_imp.to_dict(orient='records'),
    'utpm_weights': dict(zip(utpm_features, [float(w) for w in weights])),
    'utpm_lst_correlation': r_utpm_lst,
    'utpm_persistence_correlation': r_persist,
    'moran_i': moran,
    'lisa_cluster_counts': grid['lisa_cluster'].value_counts().to_dict(),
    'jenks_bins': [float(b) for b in bins],
    'jenks_class_counts': {f'class_{i}': int((classes == i).sum())
                           for i in range(JENKS_K_CLASSES)},
}
with open(UTPM_RESULTS_JSON, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f'Kaydedildi: {UTPM_RESULTS_JSON.name}')

weights_df.to_csv(TABLES / '10_utpm_weights.csv', index=False, encoding='utf-8-sig')
stats = pd.DataFrame([
    {'metric': 'UTPM × LST (r)', 'value': round(r_utpm_lst, 3)},
    {'metric': 'UTPM × Persistence (r)', 'value': round(r_persist, 3)},
    {'metric': "Global Moran's I", 'value': round(moran['I'], 4)},
    {'metric': 'Moran p', 'value': round(moran['p_value'], 4)},
    {'metric': 'Persistent HOT cells', 'value': int(ph_mask.sum())},
    {'metric': 'Jenks classes', 'value': JENKS_K_CLASSES},
])
stats.to_csv(TABLES / '10_utpm_stats.csv', index=False, encoding='utf-8-sig')
print('Kaydedildi: 10_utpm_weights.csv, 10_utpm_stats.csv')

## Özet — Tezsel Sonuç Ürünleri

1. **SHAP attribution** — RF tahminlerinin yorumlanabilir kaynak çözümlemesi
2. **UTPM lineer indeks** — Şehir planlamacılarının kullanabileceği 0-100 ölçek
3. **Jenks 5 sınıf** — Kategorik karar destek (Çok serin / Serin / Orta / Sıcak / Çok sıcak)
4. **LISA cluster** — UHI çekirdekleri (HH) ve serin adalar (LL) tespit edildi
5. **Moran's I** — UTPM mekânsal olarak kümeli (rastgele değil), UHI hipotezini istatistiksel destekler

## Sınırlılıklar

- UTPM ağırlıkları SHAP global'i — local SHAP attribution'lar hücre bazında değişir, lineer indeks bunu yansıtmıyor.
- Moran's I k-NN (k=8) seçimi yarıçap tabanlı yerine; gerçek dünya komşuluğu farklı olabilir (yol ağı vs Öklid).
- Jenks bin'ler verinin dağılımına bağlı; başka bölgede aynı sınır geçerli olmayabilir.
- UTPM dış mahallelere extrapolation güvenilir değil (Hafta 10 hold-out R²=0.10 sınırlılığı UTPM'e de yansır).

## Sonraki Adım

**Hafta 14-15** — Streamlit + Claude API prototip: kullanıcı bir adres girer → o noktanın UTPM skoru + sınıfı + cluster bilgisi + 5-yıl persistence durumu döner. 3 modüllü AI asistan (yorumlu açıklama).